In [1]:
import sys
import os
import importlib
import anndata as ad
import pandas as pd
import scanpy as sc
import logging
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

In [2]:
sys.path.append(os.path.abspath("/Users/lakshmi_nccs/Desktop/NCCS/Projects/For_Publication/HNSCC_CosMx6k/helpers/"))
import helpers
importlib.reload(helpers)
import tcell_classifier
importlib.reload(tcell_classifier)

/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/squidpy/gr/_utils.py:23: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  CAN_USE_SPARSE_ARRAY = Version(anndata.

<module 'tcell_classifier' from '/Users/lakshmi_nccs/Desktop/NCCS/Projects/For_Publication/HNSCC_CosMx6k/helpers/tcell_classifier.py'>

In [3]:
genes = ["CCL5", "CXCL8", "CXCL13", "GAL1", "GAL3", "KLF2", "LAG3", "PRF1", "PDPN"]

In [64]:
gn_name = genes[7]

In [66]:
data_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/processed_data/" + gn_name + "/"
res_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/" + gn_name + "_pp/"
if not (os.path.exists(res_dir)):
    os.makedirs(res_dir)

In [67]:
folder = Path(data_dir)
files = [f.stem for f in folder.iterdir() if f.is_file() and f.name != ".DS_Store"]

In [69]:
def preprocess_explants(fn):
    import numpy as np
    import scanpy as sc
    import anndata as ad

    # Load
    adata = ad.read_h5ad(data_dir + fn + ".h5ad")

    # -------------------------
    # 1. Total intensity filter
    # -------------------------
    adata.obs["total_intensity"] = np.array(adata.X.sum(axis=1)).flatten()

    low_cut = adata.obs["total_intensity"].quantile(0.05)
    high_cut = adata.obs["total_intensity"].quantile(0.99)

    adata = adata[
        (adata.obs["total_intensity"] > low_cut) &
        (adata.obs["total_intensity"] < high_cut)
    ].copy()

    # -------------------------
    # 2. DAPI-based filtering
    # -------------------------
    dapi = adata.to_df()["DAPI"].values

    dapi_low = np.quantile(dapi, 0.05)
    dapi_high = np.quantile(dapi, 0.995)  # conservative high cutoff

    adata = adata[(dapi > dapi_low) & (dapi < dapi_high)].copy()

    # -------------------------
    # 3. Soft DAPI normalization
    # -------------------------
    dapi = adata.to_df()["DAPI"].values
    dapi_scaled = dapi / np.median(dapi)

    adata.X = adata.X / (dapi_scaled[:, None] + 1e-6)

    # -------------------------
    # 4. Remove DAPI channel
    # -------------------------
    dapi_idx = adata.var_names.str.contains("DAPI")
    adata = adata[:, ~dapi_idx].copy()
    # -------------------------
    # 6. Save pre-normalized values
    # -------------------------
    adata.layers["preprocessed"] = adata.X.copy()

    # -------------------------
    # 8. Log transform
    # -------------------------
    sc.pp.log1p(adata)

    # -------------------------
    # 9. Store raw
    # -------------------------
    adata.raw = adata

    # -------------------------
    # 10. Save
    # -------------------------
    adata.write_h5ad(res_dir + fn + "_pp.h5ad")

    return adata

In [70]:
for fn in files:
    preprocess_explants(fn)

/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: Forward slashes will be disallowed in h5 stores in the next minor release
/opt/anaconda3/envs/bioinformatics/lib/python3.12/site-packages/anndata/_io/utils.py:243: FutureWarning: 

In [71]:
data_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/" + gn_name + "_pp/"
res_dir = "/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/" + gn_name + "_figs/"
if not (os.path.exists(res_dir)):
    os.makedirs(res_dir)
folder = Path(data_dir)    
files = [f.stem for f in folder.iterdir() if f.is_file() and f.name != ".DS_Store"]

In [73]:
def classify_calculate(fn, th1, th2):
    adata = ad.read_h5ad(data_dir+fn+".h5ad")
    adata.obs['tma_sid'] = adata.obs['filename']
    x = adata.obs["centroid_x"]
    y = adata.obs["centroid_y"]
    
    import matplotlib.pyplot as plt
    X = adata.X
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    markers = [0, 1, 2]
    for i, ax in zip(markers, axes):
        sc = ax.scatter(
            x, y,
            c=X[:, i],
            s=2,
            cmap="viridis"
        )
        ax.set_title(adata.var_names[i])
        ax.invert_yaxis()
        ax.set_xticks([])
        ax.set_yticks([])
        plt.colorbar(sc, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.savefig(res_dir + fn+ "_"+adata.obs['Patient'][0]+"_"+adata.obs['Response'][0]+"_"+adata.var_names[0]+ ".png", dpi=300, bbox_inches="tight")
    plt.close()


    from sklearn.mixture import GaussianMixture
    import numpy as np
    m0 = adata.X[:, 0]
    cd8 = adata.X[:, 1]
    panck = adata.X[:, 2]

    low = np.percentile(panck, 1)
    high = np.percentile(panck, 99)
    panck_clip = np.clip(panck, low, high)
    tumor_thresh = np.mean(panck_clip) + th1 * np.std(panck_clip)
    is_Tumor = panck_clip > tumor_thresh

    cd8_gate = cd8 > np.percentile(cd8, th2)
    #m0_gate = m0 > np.percentile(m0, th3)
    
    is_T_cell = cd8_gate & (~is_Tumor)
    #tumor_excl = panck < np.percentile(panck, 50)
    #is_T_cell_like = cd8_gate & tumor_excl
    

    #Figure 
    labels = np.full(adata.n_obs, "cells", dtype=object)
    labels[is_T_cell] = "T cell"
    labels[is_Tumor] = "Tumor"
    adata.obs["ct"] = labels
    cell_type = adata.obs["ct"]

    plt.figure(figsize=(6, 5))
    colors = {
        "Tumor": "red",
        "T cell": "blue",
        "cells": "lightgray"
    }
    for ct, color in colors.items():
        idx = cell_type == ct
        plt.scatter(x[idx], y[idx], s=2, c=color, label=ct, alpha=0.7)
    plt.gca().invert_yaxis()
    plt.xlabel("Centroid X")
    plt.ylabel("Centroid Y")
    plt.title("Spatial distribution of cell types")
    plt.legend(markerscale=3)
    plt.tight_layout()
    plt.savefig(res_dir + fn+ "_T_Cells_"+ adata.obs['Patient'][0]+"_"+ adata.obs['Response'][0]+"_"+adata.var_names[0]+ ".png", dpi=300, bbox_inches="tight")
    plt.close()

    cd8s_distance = helpers.get_query_dist_target_all(adata, "ct", "T cell", "Tumor")
    valid_genes = cd8s_distance.var_names.tolist()
    #bb = [np.arange(0, 180, 20).tolist(), max(cd8s_distance.obs['Distance_to_Tumor'])]
    #bb = list(np.arange(0, 200, 10)) + [cd8s_distance.obs["Distance_to_Tumor"].max()]
    #cd8s_distance.obs['Distance_bin'] = pd.cut(cd8s_distance.obs['Distance_to_Tumor'], bins=bb) 

    d = cd8s_distance.obs['Distance_to_Tumor']
    cd8s_distance.obs['Distance_norm'] = d / d.max()
    bins = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0]
    cd8s_distance.obs['Distance_bin'] = pd.cut(
    cd8s_distance.obs['Distance_norm'],  # or Distance_norm
    bins=bins,
    include_lowest=True)
    cd8s_distance.obs["Distance_bin"] = cd8s_distance.obs["Distance_bin"].astype(str)

    df = cd8s_distance.to_df()
    df['Distance_bin'] = cd8s_distance.obs['Distance_bin']
    df["m0_pos"] = df[adata.var_names[0]] > np.quantile(df[adata.var_names[0]], 0.5)
    summary = df.groupby('Distance_bin').agg(
        Mean_m0=(adata.var_names[0], 'mean'),
        Prop_m0=('m0_pos', 'mean'),
        Count=('m0_pos', 'size')
    ).reset_index()
    summary['Patient'] = adata.obs['Patient'][0]
    summary['Response'] = adata.obs['Response'][0]
    summary['ct'] = adata.obs['ct'][0]
    summary["rel_m0"] = summary["Mean_m0"] / summary["Mean_m0"].mean()

    plt_obj, ax = helpers.plot_gene_distance_heatmap(
    cd8s_distance,
    genes_ordered=valid_genes,
    pw=10,
    ph=8,
    x_col="Distance_bin",
    tit=adata.obs['Patient'][0]+"_"+adata.obs['Response'][0]+"_"+adata.var_names[0])

    fig = ax.get_figure()

    fig.savefig(
        res_dir + fn+ "_Distance_"+ adata.obs['Patient'][0]+"_"+ adata.obs['Response'][0]+"_"+adata.var_names[0]+ ".png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(fig)
    return(cd8s_distance)



In [74]:
files

['2026_03_24_13_19_s1_pp',
 '2026_03_24_13_29_s1_pp',
 '2026_03_24_13_13_s1_pp',
 '2026_03_17_76_s2_pp',
 '2026_03_24_13_07_s1_pp',
 '2026_03_17_12_s2_pp']

In [ ]:
#CCL5
CCL5_list = ['2026_03_17_76_s1_pp',
 '2026_03_24_13_12_s2_pp',
 '2026_03_17_12_s1_pp',
 '2026_03_24_13_09_s2_pp',
 '2026_03_24_12_57_s2_pp']

all_summaries = []
s0 = classify_calculate(CCL5_list[0], 0, 10)
s1 = classify_calculate(CCL5_list[1], 0, 20)
s2 = classify_calculate(CCL5_list[2], -0.2, 10)
s3 = classify_calculate(CCL5_list[3], 0.5, 10)
s4 = classify_calculate(CCL5_list[4], 0, 10)

all_summaries.append([s0,s1,s2,s3,s4])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")


In [ ]:
#CXCL8

CXCL8_list = ['2026_03_24_13_46_s2_pp', #HN440 NR
 '2026_03_24_13_46_s3_pp', #HN440 NR
 '2026_03_17_11_s2_pp',#P13 NR
 '2026_03_24_13_32_s2_pp', #HN431 NR
 '2026_03_24_13_12_s3_pp', #HN440 NR
 '2026_03_24_12_57_s3_pp',#HN431 NR
 '2026_03_17_77_s2_pp', #P15 R
 '2026_03_24_13_36_s2_pp'] #P8 R

all_summaries = []
s0 = classify_calculate(CXCL8_list[0], 0.2, 20)
##s1 = classify_calculate(CXCL8_list[1], 0, 10)
s2 = classify_calculate(CXCL8_list[2], 0.5, 10)
s3 = classify_calculate(CXCL8_list[3], 0, 30)
##s4 = classify_calculate(CXCL8_list[4], 0.5, 10)
##s5 = classify_calculate(CXCL8_list[5], 0.5, 10) #- might need to repeat
s6 = classify_calculate(CXCL8_list[6], 1, 20)
s7 = classify_calculate(CXCL8_list[7], -0.4, 5)

all_summaries.append([s0, s2,s3,s6,s7])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")


In [ ]:
#CXCL13
CXCL13_list = ['2026_03_24_13_29_s3_pp',
 '2026_03_24_13_32_s1_pp',
 '2026_03_17_11_s1_pp',
 '2026_03_24_13_13_s3_pp',
 '2026_03_24_13_46_s1_pp',
 '2026_03_24_13_36_s1_pp',
 '2026_03_17_77_s1_pp']

all_summaries = []
s0 = classify_calculate(CXCL13_list[0], 0.5, 10) 
#s1 = classify_calculate(CXCL13_list[1], 0, 10, 10)#HN431 done twice, we take the next one..dont want
s2 = classify_calculate(CXCL13_list[2], 0, 30)
#s3 = classify_calculate(CXCL13_list[3], 0, 30, 30) #Dont need this, we take the other HN440
s4 = classify_calculate(CXCL13_list[4], 0, 10)
s5 = classify_calculate(CXCL13_list[5], -0.2, 5)
s6 = classify_calculate(CXCL13_list[6], 0, 10)

all_summaries = []
all_summaries.append([s0,s2,s4,s5,s6])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")


In [ ]:
#GAL1
GAL1_list = ['2026_03_17_58_s1_pp',
 '2026_03_24_13_09_s1_pp',
 '2026_03_24_12_57_s1_pp',
 '2026_03_17_75_s1_pp',
 '2026_03_24_13_12_s1_pp',
  '2026_05_18_P13_S123_10_s1_pp']




all_summaries = []
s0 = classify_calculate(GAL1_list[0], 0.5, 10) 
s1 = classify_calculate(GAL1_list[1], 0, 10)
s2 = classify_calculate(GAL1_list[2], 0, 10)
s3 = classify_calculate(GAL1_list[3], 0.5, 10) 
s4 = classify_calculate(GAL1_list[4], 0, 10)
s5 = classify_calculate(GAL1_list[4], 0, 10)

all_summaries = []
all_summaries.append([s0,s1,s2,s3, s4, s5])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")


In [ ]:
#GAL3
GAL3_list = ['2026_03_17_58_s2_pp', 
             '2026_03_17_75_s2_pp',
             '2026_05_18_HN431_s1_pp',
             '2026_05_18_P13_S123_10_s2_pp',
             '2026_05_19_P13_S123_7_s2_pp']

all_summaries = []
s0 = classify_calculate(GAL3_list[0], 0.5, 10) 
s1 = classify_calculate(GAL3_list[1], 0, 20)
s2 = classify_calculate(GAL3_list[2], 0, 20)
s3 = classify_calculate(GAL3_list[3], 0, 20)
s4 = classify_calculate(GAL3_list[4], 0, 20)

all_summaries = []
all_summaries.append([s0,s1,s2,s3,s4])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")



In [ ]:
#KLF2
KLF2_list = ['2026_03_24_13_13_s2_pp', #HN440 NR
 '2026_03_24_13_29_s2_pp', #HN431 NR
 '2026_03_24_13_07_s2_pp', #HN421 R
 '2026_03_24_13_19_s2_pp', #P8 R
 '2026_05_18_P15_S123_18_s1_pp', #P15 R
 '2026_05_19_P13_S123_7_s1_pp'] #P13 NR


all_summaries = []
s0 = classify_calculate(KLF2_list[0], 0.5, 10) 
s1 = classify_calculate(KLF2_list[1], 0, 10)
s2 = classify_calculate(KLF2_list[2], 0.5, 10)
s3 = classify_calculate(KLF2_list[3], 0.8, 5)
s4 = classify_calculate(KLF2_list[4], 0.5, 10)
s5 = classify_calculate(KLF2_list[5], -0.2, 20)

all_summaries = []
all_summaries.append([s0,s1,s2,s3,s4,s5])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")


In [63]:
#LAG3
LAG3_list = ['2026_03_17_79_s1_pp', #P15 R
 '2026_03_24_13_31_s1_pp', #HN431 NR
 '2026_03_24_13_39_s1_pp', #P8 R
 '2026_03_24_13_47_s1_pp', #HN440 NR
 '2026_05_18_P15_S123_18_s2_pp', #P15 R
 '2026_05_18_HN431_s2_pp', #HN431 NR
 '2026_05_18_HN421_16_s1_pp', #HN421 R
 '2026_05_18_P13_S123_14_s1_pp'] #P13 NR

all_summaries = []
s0 = classify_calculate(LAG3_list[0], 0.2, 20) 
s1 = classify_calculate(LAG3_list[1], 0.7, 30)
s2 = classify_calculate(LAG3_list[2], 0.7, 5)
s3 = classify_calculate(LAG3_list[3], 0.5, 10)
#s4 = classify_calculate(LAG3_list[4], 0.7, 5)
#s5 = classify_calculate(LAG3_list[5], 0.5, 10)
s6 = classify_calculate(LAG3_list[6], 0.3, 5)
s7 = classify_calculate(LAG3_list[7], 0, 10)


all_summaries = []
all_summaries.append([s0,s1,s2,s3,s6,s7])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.obs["Slice_number"] = adata_all.obs["Slice_number"].astype(str)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")


/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:24: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:70: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
2026-06-25 14:04:26 - INFO - ## SAMID_FOV: 2026_03_17__79_s1.ome.tiff



🧬 Sample: 2026_03_17__79_s1.ome.tiff
Query count     : 8952
Target count : 8234


/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:98: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipyker


🧬 Sample: 2026_03_24__13_31__RecognizedCode_s1.ome.tiff
Query count     : 14141
Target count : 12022


/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:98: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipyker

...

/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:98: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipyker


🧬 Sample: 2026_05_18_HN421 D6 16-Tiles Blending_s1.ome.tiff
Query count     : 6277
Target count : 3333


/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:98: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipyker


🧬 Sample: 2026_05_18_P13D6-PEM S123 14-Tiles Blending_s1.ome.tiff
Query count     : 11610
Target count : 10860


/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:96: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:97: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipykernel_11459/2232997465.py:98: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
/var/folders/j3/kfzrbbb92cjgcnjtb19t61pm0000gn/T/ipyker

In [ ]:
#PRF1
PRF1_list = ['2026_03_24_13_19_s1_pp',
 '2026_03_24_13_29_s1_pp',
 '2026_03_24_13_13_s1_pp',
 '2026_03_17_76_s2_pp',
 '2026_03_24_13_07_s1_pp',
 '2026_03_17_12_s2_pp']

all_summaries = []
s0 = classify_calculate(PRF1_list[0], 0.5, 20)
s1 = classify_calculate(PRF1_list[1], 0.2, 10)
s2 = classify_calculate(PRF1_list[2], 1, 30)
s3 = classify_calculate(PRF1_list[3], 0.1, 5)
s4 = classify_calculate(PRF1_list[4], 0, 10)
s5 = classify_calculate(PRF1_list[5], 0.4, 10)


all_summaries.append([s0,s1,s2,s3,s4,s5])
adata_all = ad.concat(all_summaries[0], axis=0)
adata_all.write_h5ad("/Users/lakshmi_nccs/Desktop/NCCS/Projects/Explants/results/Explants_"+ gn_name + ".h5ad")
